In [12]:
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

In [13]:
CSV_PATH = Path("cov_types.csv")
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"
RANDOM_STATE = 42
SAMPLE_SIZE = 50_000

In [14]:
COLUNAS = (
    [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points",
    ]
    + [f"Wilderness_Area{i}" for i in range(1, 5)]
    + [f"Soil_Type{i}" for i in range(1, 41)]
    + ["Cover_Type"]
)

In [19]:
def carregar_base() -> pd.DataFrame:
    """Carrega cov_types.csv ou baixa a base se ela ainda nao existir."""
    if CSV_PATH.exists():
        print(f"Lendo arquivo local: {CSV_PATH.resolve()}")
        return pd.read_csv(CSV_PATH)

    print("Arquivo cov_types.csv nao encontrado.")
    print("Baixando a base oficial Covertype da UCI...")

    try:
        df = pd.read_csv(UCI_URL, header=None, names=COLUNAS, compression="gzip")
    except Exception as erro:
        raise RuntimeError(
            "Nao consegui baixar a base. Verifique sua internet ou baixe "
            "covtype.data.gz da UCI manualmente."
        ) from erro

    df.to_csv(CSV_PATH, index=False)
    print(f"Base salva em: {CSV_PATH.resolve()}")
    return df

In [20]:
def mostrar_distribuicao(y:pd.Series) -> None:
    """Mostra contagem e porcentagem de cada classe."""
    contagem = y.value_counts().sort_index()
    percentual = (y.value_counts(normalize=True).sort_index() * 100).round(2)

    tabela = pd.DataFrame({
        "Contagem": contagem,
        "Percentual (%)": percentual
    })
    print("\nDistribuição da variável alvo:\n")
    print(tabela)

    classe_majoritaria = contagem.idxmax()
    classe_majoritaria = contagem.idxmin()

    print("\nClasse majoritária:", classe_majoritaria)
    print("Classe minoritária:", classe_minoritaria)

In [21]:
def avaliar(nome:str, modelo, X_test, y_test) -> None:
    """Avalia o modelo e imprime métricas."""
    pred = modelo.predict(X_test)

    print('\n' + '='*80)
    print(nome)
    print('='*80)

    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, pred), 4))

    print("\nMatriz de Confusão:")
    print(confusion_matrix(y_test, pred))

    print("\nRelatório de Classificação:")
    print(classification_report(y_test, pred))